# 18.2 — Window dashboard

This notebook validates the completed market snapshot and canonical window-dashboard
table family before rendering two figures. It does not recompute local portfolio
metrics or write the canonical dashboard table or its German locale mirror.

Optional source overrides:
- `FINANCE_NOTEBOOK_SOURCE_ROOT`
- optional direct pin: `FINANCE_NOTEBOOK_MARKET_SNAPSHOT_DIR`


In [ ]:
import hashlib
import json
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")


def _repository_root() -> Path:
    configured = os.environ.get("FINANCE_NOTEBOOK_REPO_ROOT")
    if configured:
        candidate = Path(configured).expanduser().resolve()
        if (candidate / "pyproject.toml").exists() and (candidate / "notebooks").exists():
            return candidate
        raise ValueError(
            "FINANCE_NOTEBOOK_REPO_ROOT must name the project directory containing "
            "pyproject.toml and notebooks/"
        )
    for candidate in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent):
        if (candidate / "pyproject.toml").exists() and (candidate / "notebooks").exists():
            return candidate.resolve()
    return Path.cwd().resolve()


REPO = _repository_root()
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "scripts"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from macro_framework.reporting import report_table, validate_report_row
from scripts import build_basket_long as basket_producer
from scripts import build_tear_sheet as bts


SNAPSHOT_ID_CANDIDATES = (
    "market_total_return_fx_2026-06-30_v1",
    "provisional_market_total_return_fx_2026-06-30_v1",
)


def _resolve_override(value: str | None) -> Path | None:
    if not value:
        return None
    path = Path(value).expanduser()
    if not path.is_absolute():
        path = (REPO / path).resolve()
    return path


def _search_roots() -> list[Path]:
    override = _resolve_override(os.environ.get("FINANCE_NOTEBOOK_SOURCE_ROOT"))
    bases = [override] if override is not None else [
        REPO / "release_assets" / "data-v4",
        REPO / "data" / "provisional_remediation",
        REPO / "data",
        REPO,
    ]
    roots: list[Path] = []
    for base in bases:
        if base is None:
            continue
        roots.append(base)
        data_base = base / "data"
        if data_base != base:
            roots.append(data_base)
    deduped: list[Path] = []
    seen: set[str] = set()
    for root in roots:
        key = str(root)
        if key not in seen:
            seen.add(key)
            deduped.append(root)
    return deduped


SEARCH_ROOTS = _search_roots()


def pretty_path(path: Path) -> str:
    resolved = path.resolve()
    try:
        return str(resolved.relative_to(REPO))
    except ValueError:
        return str(resolved)


def find_existing_path(relative_candidates: list[str | Path]) -> Path | None:
    rels = [Path(rel) for rel in relative_candidates]
    for root in SEARCH_ROOTS:
        for rel in rels:
            candidate = root / rel
            if candidate.exists():
                return candidate
    return None


def _find_path(relative_candidates: list[str | Path]) -> Path:
    path = find_existing_path(relative_candidates)
    if path is not None:
        return path
    rels = [Path(rel) for rel in relative_candidates]
    return SEARCH_ROOTS[0] / rels[0]


def find_table_path(stem: str, *, prefer: str = "table") -> Path | None:
    candidates = {
        "table": [
            Path("tables") / f"{stem}.parquet",
            Path("mirrors") / f"{stem}.csv",
            Path("tear_sheet") / f"{stem}.csv",
        ],
        "mirror": [
            Path("mirrors") / f"{stem}.csv",
            Path("tear_sheet") / f"{stem}.csv",
            Path("tables") / f"{stem}.parquet",
        ],
        "german": [
            Path("mirrors") / f"{stem}_de.csv",
            Path("tear_sheet") / f"{stem}_de.csv",
        ],
    }
    return find_existing_path(candidates[prefer])


def load_frame(path: Path, **csv_kwargs) -> pd.DataFrame:
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    if path.suffix == ".csv":
        options = {"comment": "#"}
        options.update(csv_kwargs)
        return pd.read_csv(path, **options)
    raise ValueError(f"unsupported table file type: {path}")


def resolve_snapshot_dir() -> Path:
    override = _resolve_override(os.environ.get("FINANCE_NOTEBOOK_MARKET_SNAPSHOT_DIR"))
    if override is not None:
        return override
    rels = []
    for snapshot_id in SNAPSHOT_ID_CANDIDATES:
        rels.extend(
            [
                Path("market_snapshots") / snapshot_id,
                Path("provisional_remediation") / "market_snapshots" / snapshot_id,
            ]
        )
    return _find_path(rels)


def sha256_file(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def read_manifest(run_dir: Path) -> dict[str, object]:
    return json.loads((run_dir / "manifest.json").read_text())


def load_market_input(snapshot_dir: Path):
    manifest = read_manifest(snapshot_dir)
    manifest_sha256 = sha256_file(snapshot_dir / "manifest.json")
    basket_producer.validate_market_snapshot(snapshot_dir)
    market_input = bts.load_market_report_input(
        snapshot_dir,
        snapshot_id=manifest["snapshot_id"],
        manifest_sha256=manifest_sha256,
    )
    return market_input, manifest, manifest_sha256


_DATE_COLUMNS = {
    "start",
    "end",
    "actual_end",
    "anchor",
    "first_return_date",
    "requested_start",
    "requested_end",
    "raw_market_model_start",
    "raw_market_model_end",
}


def normalize_table(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    for column in out.columns:
        if column in _DATE_COLUMNS or column.endswith("_date"):
            out[column] = pd.to_datetime(out[column], format="mixed", errors="coerce")
    return out


def _unpadded_row(raw: dict[str, object]) -> dict[str, object]:
    return {
        key: value
        for key, value in raw.items()
        if value is not None
        and not (
            not isinstance(value, (str, bytes))
            and bool(pd.isna(value))
        )
    }


def dashboard_plot_view(frame: pd.DataFrame) -> pd.DataFrame:
    checked = normalize_table(frame).copy()
    readers = checked[checked["schema"] == "portfolio_metrics.reader.v2"].copy()
    attrs = checked[checked["schema"] == "attribution.raw_market_model.v1"].copy()
    attr_by_key: dict[tuple[str, str], dict[str, object]] = {}
    for raw in attrs.to_dict(orient="records"):
        cleaned = validate_report_row(_unpadded_row(raw))
        attr_by_key[(cleaned["portfolio_id"], cleaned["window_label"])] = cleaned

    rows: list[dict[str, object]] = []
    for raw in readers.to_dict(orient="records"):
        cleaned = validate_report_row(_unpadded_row(raw))
        assert cleaned["row_kind"] == "full", cleaned["window_label"]
        key = (cleaned["portfolio_id"], cleaned["window_label"])
        assert key in attr_by_key, f"missing canonical attribution row for {key}"
        attr = attr_by_key[key]
        assert pd.Timestamp(cleaned["start"]) == pd.Timestamp(attr["raw_market_model_start"])
        assert pd.Timestamp(cleaned["end"]) == pd.Timestamp(attr["raw_market_model_end"])
        assert int(cleaned["n_obs"]) == int(attr["raw_market_model_n_obs"])
        r2 = float(cleaned["raw_market_model_r2"])
        residual_vol = float(cleaned["ann_vol"]) * float(max(0.0, 1.0 - r2) ** 0.5)
        appraisal = (
            float(cleaned["raw_market_model_intercept_ann_arithmetic"]) / residual_vol
            if residual_vol > 0.0
            else np.nan
        )
        rows.append(
            {
                "portfolio_id": cleaned["portfolio_id"],
                "window_label": cleaned["window_label"],
                "start": pd.Timestamp(cleaned["start"]),
                "end": pd.Timestamp(cleaned["end"]),
                "years": (
                    pd.Timestamp(cleaned["end"]) - pd.Timestamp(cleaned["start"])
                ).days / 365.25,
                "n_obs": int(cleaned["n_obs"]),
                "total_return": float(cleaned["total_return"]),
                "cagr": float(cleaned["cagr"]),
                "ann_vol": float(cleaned["ann_vol"]),
                "cagr_over_vol": float(cleaned["cagr"]) / float(cleaned["ann_vol"]),
                "sharpe": float(cleaned["sharpe"]),
                "maxdd": float(cleaned["maxdd"]),
                "calmar": float(cleaned["calmar"]),
                "alpha_ann": float(cleaned["raw_market_model_intercept_ann_arithmetic"]),
                "residual_vol_ann": residual_vol,
                "appraisal": appraisal,
                "hac_t": float(cleaned["raw_market_model_intercept_t_hac"]),
                "r2": r2,
                "regression_n_obs": int(cleaned["raw_market_model_n_obs"]),
                "raw_market_model_start": pd.Timestamp(cleaned["raw_market_model_start"]),
                "raw_market_model_end": pd.Timestamp(cleaned["raw_market_model_end"]),
                "raw_market_model_n_obs": int(cleaned["raw_market_model_n_obs"]),
                "ssr": float(cleaned["ssr_ssr"]),
                "cash_benchmark_id": str(cleaned["cash_benchmark_id"]),
                "currency_basis": str(cleaned["currency_basis"]),
                "source": str(cleaned["source"]),
                "benchmark_source": str(attr["source"]),
            }
        )
    return pd.DataFrame(rows)

In [ ]:
MARKET_SNAPSHOT_DIR = resolve_snapshot_dir()
market_input, market_manifest, market_sha = load_market_input(MARKET_SNAPSHOT_DIR)

REPORT_ROOT = _resolve_override(os.environ.get("FINANCE_NOTEBOOK_REPORT_ROOT"))
if REPORT_ROOT is None:
    raise ValueError("set FINANCE_NOTEBOOK_REPORT_ROOT to a completed canonical_reports.v1 directory")
OUTPUT_DIR = _resolve_override(os.environ.get("FINANCE_NOTEBOOK_OUTPUT_DIR")) or (REPO / "data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
report_manifest_path = REPORT_ROOT / "manifest.json"
report_completed_path = REPORT_ROOT / "COMPLETED"
assert report_manifest_path.is_file() and report_completed_path.is_file()
report_manifest = json.loads(report_manifest_path.read_text())
report_manifest_sha = sha256_file(report_manifest_path)
assert report_manifest.get("schema") == "canonical_reports.v1"
assert report_manifest.get("completed") is True
assert f"manifest_sha256={report_manifest_sha}" in report_completed_path.read_text().splitlines()
assert report_manifest["input_manifests"]["market_snapshot"]["snapshot_id"] == market_manifest["snapshot_id"]
assert report_manifest["input_manifests"]["market_snapshot"]["manifest_sha256"] == market_sha

STATIC_WINDOWS = (
    bts.StaticWindowSpec("Full 16.7y (buy 2009)", pd.Timestamp("2009-09-25"), pd.Timestamp("2026-05-29")),
    bts.StaticWindowSpec("16.4y (buy 2009)", pd.Timestamp("2009-09-25"), pd.Timestamp("2026-01-30")),
    bts.StaticWindowSpec("10.0y (buy 2016)", pd.Timestamp("2016-02-01"), pd.Timestamp("2026-01-30")),
    bts.StaticWindowSpec("7.1y (buy 2019)", pd.Timestamp("2019-01-02"), pd.Timestamp("2026-01-30")),
)
PLOT_ORDER = (
    "7.1y (buy 2019)",
    "10.0y (buy 2016)",
    "16.4y (buy 2009)",
    "Full 16.7y (buy 2009)",
)
WINDOW_STYLE = {
    "7.1y (buy 2019)": {"color": "#e8710a", "marker": "o"},
    "10.0y (buy 2016)": {"color": "#2f6db3", "marker": "o"},
    "16.4y (buy 2009)": {"color": "#8656c9", "marker": "o"},
    "Full 16.7y (buy 2009)": {"color": "#4a4a4a", "marker": "D"},
}

expected_rows = []
for spec in STATIC_WINDOWS:
    reader_row, attribution_row = bts.build_static_bh_rows(
        market_input,
        spec,
        attribution=True,
    )
    expected_rows.append(reader_row)
    expected_rows.append(attribution_row)
expected_dashboard = report_table([row for row in expected_rows if row is not None])

entry = report_manifest["tables"].get("tear_sheet_static_bh_window_dashboard")
entry_de = report_manifest["mirrors"].get("tear_sheet_static_bh_window_dashboard_de.csv")
assert isinstance(entry, dict), "canonical window-dashboard table is missing"
assert isinstance(entry_de, dict), "canonical German window-dashboard mirror is missing"
dashboard_path = (REPORT_ROOT / entry["file"]).resolve()
dashboard_de_path = (REPORT_ROOT / entry_de["file"]).resolve()
assert dashboard_path.is_relative_to(REPORT_ROOT.resolve())
assert dashboard_de_path.is_relative_to(REPORT_ROOT.resolve())
assert dashboard_path.is_file() and sha256_file(dashboard_path) == entry["sha256"]
assert dashboard_de_path.is_file() and sha256_file(dashboard_de_path) == entry_de["sha256"]

dashboard_loaded = pd.read_parquet(dashboard_path)
dashboard_de_loaded = pd.read_csv(dashboard_de_path, sep=";", decimal=",")

sort_keys = ["window_label", "schema"]
loaded_sorted = dashboard_loaded.sort_values(sort_keys, kind="stable").reset_index(drop=True)
expected_sorted = expected_dashboard.sort_values(sort_keys, kind="stable").reset_index(drop=True)
de_sorted = dashboard_de_loaded.sort_values(sort_keys, kind="stable").reset_index(drop=True)
pd.testing.assert_frame_equal(
    normalize_table(loaded_sorted[expected_sorted.columns]),
    normalize_table(expected_sorted),
    check_dtype=False,
)
pd.testing.assert_frame_equal(
    normalize_table(de_sorted[expected_sorted.columns]),
    normalize_table(expected_sorted),
    check_dtype=False,
)

window_rows = dashboard_plot_view(dashboard_loaded).set_index("window_label").loc[list(PLOT_ORDER)].reset_index()
expected_reader_by_label = {
    row["window_label"]: row
    for row in expected_dashboard[expected_dashboard["schema"] == "portfolio_metrics.reader.v2"].to_dict(orient="records")
}

basket_sha = market_manifest["files"]["basket_adjusted_close_local.parquet"]["sha256"]
benchmark_sha = market_manifest["files"]["cash_market_total_return.parquet"]["sha256"]
for spec in STATIC_WINDOWS:
    row = window_rows[window_rows["window_label"] == spec.label].iloc[0]
    expected_row = expected_reader_by_label[spec.label]
    assert pd.Timestamp(row["start"]) == pd.Timestamp(expected_row["start"])
    assert pd.Timestamp(row["end"]) == pd.Timestamp(expected_row["end"])
    assert int(row["n_obs"]) == int(expected_row["n_obs"])
    assert row["cash_benchmark_id"] == f"BIL@{market_manifest['snapshot_id']}"
    assert row["currency_basis"] == "legacy_mixed_local_quotes"
    assert f"market_snapshot:{market_manifest['snapshot_id']}/basket_adjusted_close_local.parquet#{basket_sha}" in row["source"]
    assert f"market_snapshot:{market_manifest['snapshot_id']}/cash_market_total_return.parquet#BIL@{benchmark_sha}" in row["source"]
    assert f"market_snapshot:{market_manifest['snapshot_id']}/cash_market_total_return.parquet#SPY@{benchmark_sha}" in row["benchmark_source"]
    assert pd.Timestamp(row["raw_market_model_start"]) == pd.Timestamp(row["start"])
    assert pd.Timestamp(row["raw_market_model_end"]) == pd.Timestamp(row["end"])
    assert int(row["raw_market_model_n_obs"]) == int(row["n_obs"])
    for field in (
        "cagr",
        "ann_vol",
        "cagr_over_vol",
        "sharpe",
        "calmar",
        "alpha_ann",
        "residual_vol_ann",
        "appraisal",
        "r2",
        "ssr",
    ):
        assert np.isfinite(float(row[field])), (spec.label, field, row[field])

display(
    window_rows[
        [
            "window_label",
            "start",
            "end",
            "n_obs",
            "total_return",
            "cagr",
            "ann_vol",
            "sharpe",
            "maxdd",
            "calmar",
            "alpha_ann",
            "residual_vol_ann",
            "appraisal",
            "ssr",
        ]
    ]
)
print("market snapshot:", market_manifest["snapshot_id"], market_sha)
print("canonical report root:", REPORT_ROOT, report_manifest_sha)
print("canonical dashboard table:", pretty_path(dashboard_path), sha256_file(dashboard_path))
print("canonical dashboard mirror:", pretty_path(dashboard_de_path), sha256_file(dashboard_de_path))

## 1. Risk–return map — one point per window

Each point is the same static basket over a different validated window. Gray rays
are return-per-volatility reference slopes. The dashed path connects common-end
windows from the shortest sample to the longest.


In [ ]:
plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
INK, MUTED = "#333333", "#8a8a8a"


def ray_panel(ax, slopes, slope_fmt, xmax, ymax, frac):
    ax.set_xlim(0, xmax)
    ax.set_ylim(0, ymax)
    ax.grid(alpha=0.25, zorder=0)
    for slope in slopes:
        ax.plot([0, xmax], [0, slope * xmax], color="#cccccc", lw=1, zorder=1)
        x_end = min(xmax, ymax / slope)
        p0 = ax.transData.transform((0.0, 0.0))
        p1 = ax.transData.transform((x_end, slope * x_end))
        angle = np.degrees(np.arctan2(p1[1] - p0[1], p1[0] - p0[0]))
        ax.annotate(
            slope_fmt.format(slope),
            (frac * x_end, frac * slope * x_end),
            color=MUTED,
            fontsize=8,
            ha="center",
            va="bottom",
            rotation=angle,
            rotation_mode="anchor",
            xytext=(0, 2),
            textcoords="offset points",
        )


fig, ax = plt.subplots(figsize=(9.0, 5.4))
ray_panel(ax, (0.8, 1.0, 1.2, 1.4), "CAGR/σ = {:.1f}", 18.0, 19.5, frac=0.60)

walk = ["7.1y (buy 2019)", "10.0y (buy 2016)", "16.4y (buy 2009)"]
row_by_label = window_rows.set_index("window_label")
ax.plot(
    [float(row_by_label.loc[label, "ann_vol"]) * 100.0 for label in walk],
    [float(row_by_label.loc[label, "cagr"]) * 100.0 for label in walk],
    ls="--",
    lw=1.4,
    color="#999999",
    zorder=2,
    label="window extended backwards: 7.1y → 10.0y → 16.4y",
)
ax.annotate(
    "",
    xy=(float(row_by_label.loc[walk[-1], "ann_vol"]) * 100.0, float(row_by_label.loc[walk[-1], "cagr"]) * 100.0),
    xytext=(float(row_by_label.loc[walk[-2], "ann_vol"]) * 100.0, float(row_by_label.loc[walk[-2], "cagr"]) * 100.0),
    arrowprops=dict(arrowstyle="-|>", color="#999999", lw=1.4),
    zorder=2,
)

offsets = {
    "7.1y (buy 2019)": (11, 3),
    "10.0y (buy 2016)": (11, 3),
    "16.4y (buy 2009)": (11, -13),
    "Full 16.7y (buy 2009)": (11, 3),
}
for label in PLOT_ORDER:
    row = row_by_label.loc[label]
    style = WINDOW_STYLE[label]
    x = float(row["ann_vol"]) * 100.0
    y = float(row["cagr"]) * 100.0
    ax.scatter(
        x,
        y,
        s=170 if style["marker"] == "o" else 130,
        marker=style["marker"],
        color=style["color"],
        edgecolor="white",
        linewidth=1.6,
        zorder=4,
        label=label,
    )
    ax.annotate(
        f"{label}\nCAGR/σ {row['cagr_over_vol']:.2f}",
        (x, y),
        xytext=offsets[label],
        textcoords="offset points",
        fontsize=8.5,
        color=INK,
        ha="left",
    )

ax.set_xlabel("annualized volatility σ (%)")
ax.set_ylabel("CAGR (%)")
ax.set_title(
    "One static buy-and-hold line, four validated windows — steeper ray = better\n"
    "(diamond = the only rung with the later 2026-05-29 end)",
    fontsize=11,
)
ax.legend(loc="upper left", frameon=False, fontsize=9, title="observation window", title_fontsize=9)
fig.tight_layout()
risk_return_path = OUTPUT_DIR / "nb18_2_risk_return_map.png"
fig.savefig(risk_return_path, dpi=300, bbox_inches="tight")
plt.show()

print("figure:", risk_return_path)

## 2. Ratio ladder — three denominators, one window family

Each row divides return by a different risk measure. The ordering comes
straight from the validated canonical rows and their matched benchmark
lineage.


In [ ]:
ratios = [
    ("Sharpe (return / total vol)", "sharpe"),
    ("Calmar (CAGR / |max drawdown|)", "calmar"),
    ("Appraisal (regression intercept / residual vol)", "appraisal"),
]
slot = {
    "7.1y (buy 2019)": 11,
    "10.0y (buy 2016)": 11,
    "16.4y (buy 2009)": -17,
    "Full 16.7y (buy 2009)": 11,
}

fig, ax = plt.subplots(figsize=(9.5, 3.9))
for i, (row_label, key) in enumerate(ratios):
    y = len(ratios) - 1 - i
    xs = [float(row_by_label.loc[label, key]) for label in PLOT_ORDER]
    ax.plot([min(xs), max(xs)], [y, y], color="#d5d5d5", lw=2, zorder=1)
    for label in PLOT_ORDER:
        row = row_by_label.loc[label]
        style = WINDOW_STYLE[label]
        ax.scatter(
            float(row[key]),
            y,
            s=150 if style["marker"] == "o" else 115,
            marker=style["marker"],
            color=style["color"],
            edgecolor="white",
            linewidth=1.5,
            zorder=3,
            label=label if i == 0 else None,
        )
        ax.annotate(
            f"{float(row[key]):.2f}",
            (float(row[key]), y),
            xytext=(0, slot[label]),
            textcoords="offset points",
            ha="center",
            fontsize=8.5,
            color=INK,
        )

all_values = [float(row_by_label.loc[label, key]) for label in PLOT_ORDER for _, key in ratios]
pad = 0.18 * (max(all_values) - min(all_values))
ax.set_yticks(range(len(ratios)), [label for label, _ in reversed(ratios)])
ax.set_xlabel("return per unit of risk (unitless ratio)")
ax.set_xlim(min(all_values) - pad, max(all_values) + pad)
ax.set_ylim(-0.75, 2.75)
ax.grid(axis="x", alpha=0.25, zorder=0)
ax.legend(loc="upper left", frameon=False, fontsize=9, ncol=2)
ax.set_title(
    "Return per unit of risk, by observation window — all values come from validated canonical rows",
    fontsize=11,
    pad=14,
)
fig.tight_layout()
ratio_path = OUTPUT_DIR / "nb18_2_ratio_ladder.png"
fig.savefig(ratio_path, dpi=300, bbox_inches="tight")
plt.show()

print("figure:", ratio_path)

## 3. Canonical dashboard rows and provenance checks

The figures above consume only the validated canonical dashboard family.
The notebook keeps the locale mirrors read-only and projects no new
canonical table.


In [ ]:
display(
    window_rows[
        [
            "window_label",
            "start",
            "end",
            "years",
            "n_obs",
            "total_return",
            "cagr",
            "ann_vol",
            "cagr_over_vol",
            "sharpe",
            "maxdd",
            "calmar",
            "alpha_ann",
            "residual_vol_ann",
            "appraisal",
            "ssr",
        ]
    ]
)

print("reader row source hashes and benchmark hashes are pinned before plotting")
for label in PLOT_ORDER:
    row = row_by_label.loc[label]
    print(
        f"{label}: {pd.Timestamp(row['start']):%Y-%m-%d} → {pd.Timestamp(row['end']):%Y-%m-%d}, "
        f"n={int(row['n_obs'])}, source={row['source']}"
    )
    print(f"    benchmark source: {row['benchmark_source']}")

In [ ]:
short = row_by_label.loc["7.1y (buy 2019)"]
long_window = row_by_label.loc["16.4y (buy 2009)"]
print("Same basket, same construction, same 2026-01-30 end — only the buy date moves.")
print(f"{'':<34s}{'7.1y (2019)':>12s}{'16.4y (2009)':>14s}{'16.4y / 7.1y':>14s}")
for name, key, good in [
    ("CAGR", "cagr", True),
    ("Annualized volatility σ", "ann_vol", False),
    ("Max drawdown", "maxdd", False),
    ("Sharpe", "sharpe", True),
    ("Calmar", "calmar", True),
    ("Appraisal", "appraisal", True),
    ("Regression intercept (ann.)", "alpha_ann", True),
]:
    ratio = float(long_window[key]) / float(short[key])
    tag = "" if good else "  (risk: >100% = worse)"
    print(f"  {name:<32s}{float(short[key]):>12.4f}{float(long_window[key]):>14.4f}{ratio:>13.0%}{tag}")
print(
    f"\n  Sharpe ladder: {float(short['sharpe']):.2f} (7.1y) → "
    f"{float(row_by_label.loc['10.0y (buy 2016)', 'sharpe']):.2f} (10.0y) → "
    f"{float(long_window['sharpe']):.2f} (16.4y)"
)


## Reading guide

- Points are windows, not strategies. Each figure shows one static basket through
  four validated start dates.
- The figures use one completed snapshot lineage and one canonical dashboard family.
  Dates, counts, cash benchmark identity, currency basis, and source hashes are
  checked before rendering.
- Residual volatility and appraisal come from the published raw market-model
  intercept and R² fields; the notebook does not calculate an alternative private
  metric path.
- The notebook writes only `data/nb18_2_risk_return_map.png` and
  `data/nb18_2_ratio_ladder.png`.
